In [0]:
import sys

sys.path.insert(
    0,
    "/Workspace/Users/devanshi.mann@thoughtswinsystems.com/retailpulse"
)

from retail_data_generator import Generator, release

print("Generator imported successfully!")

In [0]:
OUT = "/Volumes/workspace/rp_landing/raw"

gen = Generator(seed=42)
gen.generate(OUT)

print("RetailPulse data generated successfully!")

In [0]:
release(OUT, days=7)

In [0]:
display(
    dbutils.fs.ls(
        OUT + "/landing/sales_daily"
    )
)

In [0]:
source_path = OUT + "/landing/sales_daily"

schema_path = OUT + "/ops/schema/sales_daily_smoke"

checkpoint_path = OUT + "/ops/chk/sales_daily_smoke"

print("Source:", source_path)
print("Schema:", schema_path)
print("Checkpoint:", checkpoint_path)

In [0]:
sales_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
)

display(
    sales_stream,
    checkpointLocation=checkpoint_path
)

In [0]:

RAW = "/Volumes/workspace/rp_landing/raw"

source_path = RAW + "/landing/sales_daily"
schema_path = RAW + "/ops/schemas/m0_autoloader_test"
checkpoint_path = RAW + "/ops/chk/m0_autoloader_test"

test_table = "workspace.rp_bronze.m0_autoloader_test"

print(source_path)
print(schema_path)
print(checkpoint_path)


In [0]:

from pyspark.sql import functions as F

test_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .load(source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
        F.current_timestamp().alias("_ingest_ts")
    )
)

query = (
    test_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(test_table)
)

display(spark.table(test_table))


In [0]:
%sql
SHOW GRANTS ON SCHEMA workspace.rp_bronze;
